# Lexer e parser da linguagem LARP

Analisador lexico (Flex) e sintatico (Bison) para uma linguagem em estilo C com
declaracoes, atribuicoes, `if`/`else`, ternario (`if (...) -> ... else -> ...`) e
`route`/`fall`.

- O **lexer** (`clarker.l`) reconhece os tokens, ignora os espacos em branco e reporta
  com o numero da linha os erros lexicos.
- O **parser** (`parser.y`) recebe os tokens do lexer e verifica a estrutura do programa.
  Imprime `Programa sintaticamente correto` ou o primeiro erro sintatico, com a linha,
  alem dos erros lexicos encontrados no caminho.

O lexer tambem pode rodar sozinho: nesse caso imprime a sequencia de tokens, a tabela de
simbolos e um resumo.

## 1. Clonar o repositorio

In [ ]:
import os

REPO = 'https://github.com/Rafaeruo/parser-larp.git'
DIR = '/content/parser-larp'

if not os.path.exists(DIR):
    !git clone $REPO $DIR

%cd $DIR
!git pull --quiet && echo 'repositorio atualizado'

!apt-get -qq install -y tree
!tree

## 2. Instalar o Flex e o Bison e compilar

O Colab ja vem com `gcc`, `g++` e `make`; apenas o `flex` e o `bison` precisam ser
instalados. Sao gerados dois binarios: `bin/parser` (lexer + parser) e `bin/lexer`
(so o lexer). Se a gramatica tiver conflitos, o Bison avisa aqui na compilacao.

In [ ]:
!apt-get -qq install -y flex bison
!make
!ls -l bin/

## 3. Rodar o parser em todos os programas de teste

Passa o parser em cada arquivo de `test-programs/`. A dezena do numero do arquivo indica
o que o teste pretende exercitar:

| Faixa | Categoria        | Saida esperada do parser                                |
|-------|------------------|---------------------------------------------------------|
| `0x`  | validos          | `Programa sintaticamente correto`                       |
| `1x`  | erros lexicos    | erros lexicos ate o primeiro erro sintatico (cascatas)  |
| `2x`  | erros sintaticos | um erro sintatico, com a linha                          |

In [ ]:
!make test

## 4. Rodar so o lexer em todos os programas de teste

Roda os mesmo arquivos mas só com o lexer. Os testes `2x` passam nesse caso com 0 erros, já que
sao lexicamente validos.

In [ ]:
!make test-lexer

## 5. Rodar um programa específico

Mostra o programa, a saida do parser e a sequencia de tokens do lexer. Basta trocar o
`ARQUIVO`.

In [ ]:
ARQUIVO = 'test-programs/06-if-else.larp'

!cat $ARQUIVO
print('-' * 60)
!make run FILE=$ARQUIVO
print('-' * 60)
!make run-lexer FILE=$ARQUIVO

O parser aponta a linha e o que era esperado em um programa com erro sintatico:

In [ ]:
ARQUIVO = 'test-programs/25-if-sem-parenteses.larp'

!cat $ARQUIVO
print('-' * 60)
!make run FILE=$ARQUIVO

## 6. Criar um novo programa de teste

Exemplo para criar um novo programa de teste para testar aqui imediatamente.

In [ ]:
%%writefile test-programs/exemplo.larp
int limite, faixa;
float media;

limite = 10 + 5 * 2;
faixa = limite / 10;

if (limite > 20) {
    media = 1.5;
} else {
    media = 0.5;
}

route faixa to {
    1, 2: { limite = limite + 1; fall; }
    3: { limite = 0; }
} else {
    limite = 1;
}

In [ ]:
!make run FILE=test-programs/exemplo.larp
print('-' * 60)
!make run-lexer FILE=test-programs/exemplo.larp